In [1]:
!pip install FLAML > /dev/null

In [2]:
import numpy as np
import pandas as pd
from flaml import AutoML

# Загрузка и предобработка

In [3]:
train = pd.read_csv("/kaggle/input/advanced-dls-spring-2021/train.csv")
train2 = pd.read_csv("/kaggle/input/dfghdfgdfh/churn.csv")
test = pd.read_csv("/kaggle/input/advanced-dls-spring-2021/test.csv")

data = pd.concat([train, train2], ignore_index=True)

data["TotalSpent"] = data["TotalSpent"].replace(" ", np.nan).fillna(0).astype(float)
test["TotalSpent"] = test["TotalSpent"].replace(" ", np.nan).fillna(0).astype(float)

patterns = {
    "No": 0, "No internet service": 0, "No phone service": 0, "Yes": 1,
    "Male": 0, "Female": 1,
    "DSL": 1, "Fiber optic": 2,
    "Month-to-month": 0, "One year": 1, "Two year": 2,
    "Credit card (automatic)": 0, "Bank transfer (automatic)": 1,
    "Mailed check": 2, "Electronic check": 3,
}

target = "Churn"

X_train = data.drop(columns=[target]).replace(patterns)
X_test = test.replace(patterns)

y_train = data[target]

X_train["AvgSpent"] = X_train["TotalSpent"] / (X_train["ClientPeriod"] + 1e-12)
X_test["AvgSpent"] = X_test["TotalSpent"] / (X_test["ClientPeriod"] + 1e-12)

/tmp/ipykernel_13/3522945682.py:21: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_train = data.drop(columns=[target]).replace(patterns)
/tmp/ipykernel_13/3522945682.py:22: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  X_test = test.replace(patterns)


# FLAML (Fast and Lightweight AutoML) — это библиотека автоматического машинного обучения (AutoML), разработанная Microsoft Research.

## FLAML автоматически пробует разные модели:
    - Logistic Regression
    - Random Forest
    - XGBoost
    - LightGBM
    - CatBoost
    - ExtraTrees и др.

In [4]:
automl = AutoML()

automl.fit(
    X_train, y_train,
    time_budget=500,
    metric='roc_auc',
    task='classification',
    eval_method='cv',
    n_splits=4,
    seed=43,
    log_file_name=None,
    verbose=0
)
preds = automl.predict_proba(X_test)[:, 1]

In [5]:
# Сохранение
submission = pd.read_csv("/kaggle/input/advanced-dls-spring-2021/submission.csv")

submission["Churn"] = preds
submission.to_csv("submission.csv", index=False)
submission.head()

,Id,Churn
0,0,0.0
1,1,1.0
2,2,0.0
3,3,0.0
4,4,0.0
